In [ ]:
import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
"""
ACF Periodicity Correlation Analysis
=====================================
Harmonizes the clinical data table (Caspr1 & PanNF sheets),
parses mean ± SD from embedded strings, and runs
Spearman correlations between ACF periodicity distances
and clinical variables.

Usage:
    python acf_correlation_analysis.py
    # or paste cells into a Jupyter notebook
"""



# ── 0. Load ──────────────────────────────────────────────────────────────────
FILE = "../data/metadata/acf_harmonised.xlsx"          # adjust path if needed

sheets_raw = pd.read_excel(FILE, sheet_name=None)

# ── 1. Column name harmonisation map ────────────────────────────────────────
COL_MAP = {
    "Patients ":                                                       "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":                                                   "diabetes_type",
    "age":                                                             "age",
    "sex":                                                             "sex",
    "Duration of Disease (months)":                                    "disease_duration_months",
    "SNAP (sural nerve)":                                              "SNAP_sural",
    "HbA1c (%)":                                                       "HbA1c_pct",
    "NIS-LL":                                                          "NIS_LL",
    "SAS":                                                             "SAS",
    "NPSI":                                                            "NPSI",
    "IENFD(per mm)":                                                   "IENFD_per_mm",
    # sheet-specific periodicity columns  (handled separately below)
}

In [ ]:


# ── 2. Helper: fix European decimal commas and parse to float ────────────────
def fix_decimal(val):
    """Replace comma-decimal (3,1 → 3.1) and convert to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# ── 3. Helper: parse "330.02(±160)" → (mean, sd) ────────────────────────────
_PATTERN = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)"   # mean part
    r"\s*\(\s*[±+\-]?\s*"            # opening paren + optional ± sign
    r"([0-9]+(?:[.,][0-9]+)?)"       # SD part
    r"\s*\)\s*$"                     # closing paren
)

def parse_mean_sd(cell):
    """
    Returns (mean_float, sd_float) from strings like:
        '330.02(±160)'   '314.86 (±156)'   '456(±153)'
    Returns (NaN, NaN) if parsing fails.
    """
    if pd.isna(cell):
        return np.nan, np.nan
    m = _PATTERN.match(str(cell).strip())
    if m:
        mean = float(m.group(1).replace(",", "."))
        sd   = float(m.group(2).replace(",", "."))
        return mean, sd
    # fallback: try plain number (no SD given)
    try:
        return float(str(cell).replace(",", ".")), np.nan
    except ValueError:
        return np.nan, np.nan

# ── 4. Process each sheet ────────────────────────────────────────────────────
NUMERIC_COLS = [
    "age", "disease_duration_months", "SNAP_sural",
    "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm",
]

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}

frames = {}
for sheet_name, df in sheets_raw.items():
    if sheet_name not in PERIODICITY_RAW:
        continue

    # drop unnamed helper columns
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # rename known columns
    df = df.rename(columns=COL_MAP)

    # fix group column: strip text annotations like "1 (SFN)"
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)

    # fix sex: binary encode
    df["sex_male"] = (df["sex"].astype(str).str.lower().str.strip() == "male").astype(int)

    # fix numeric columns with possible comma-decimals
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = df[col].apply(fix_decimal)

    # parse periodicity mean ± SD
    raw_col = PERIODICITY_RAW[sheet_name]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"]  = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]    = [p[1] for p in parsed]
        df["acf_peak_cv"]       = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]  # coefficient of variation
        df = df.drop(columns=[raw_col])

    frames[sheet_name] = df
    print(f"\n=== {sheet_name}: {df.shape[0]} rows, {df.shape[1]} cols ===")
    print(df[["patient_id", "group", "age", "acf_peak_mean_nm", "acf_peak_sd_nm"]].to_string(index=False))

# ── 5. Combine for joint analysis ────────────────────────────────────────────
for name, df in frames.items():
    df["marker"] = name

combined = pd.concat(frames.values(), ignore_index=True)

# ── 6. Spearman correlation: ACF peak mean vs. clinical variables ────────────
CLINICAL = ["age", "disease_duration_months", "SNAP_sural",
            "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm"]

print("\n\n=== Spearman correlations: acf_peak_mean_nm vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)

results = []
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_mean_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_mean_nm"], sub[clin])
        results.append(dict(
            marker=sheet_name, variable=clin,
            rho=rho, pval=pval, n=len(sub)
        ))
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

results_df = pd.DataFrame(results)

# ── 7. Why SD/variance of the ACF peak matters ───────────────────────────────
print("""
=============================================================
WHY VARIANCE (SD) OF THE ACF PEAK DISTANCE MATTERS
=============================================================

1. BIOLOGICAL NOISE vs. TRUE SIGNAL
   The SD you already stored reflects within-axon variability of
   node-of-Ranvier spacing.  A large SD means the periodicity is
   irregular — pathological remodelling, demyelination, or
   measurement noise.  Using only the mean ignores this.

2. CORRELATION POWER
   If two patients share the same mean periodicity but very different
   SDs, their nerve physiology differs.  Regressing mean alone against,
   e.g., NIS-LL will leave variance-driven variance unexplained,
   inflating residuals and reducing statistical power.

3. SEPARATE PREDICTORS
   acf_peak_sd_nm (or CV = SD/mean) is itself a clinical correlate.
   It may correlate with disease severity (NIS-LL, NPSI) independently
   of the mean, because irregular spacing disrupts saltatory conduction
   even when average spacing is normal.

4. WEIGHTED CORRELATIONS
   If SD differs greatly across patients, Pearson/Spearman on the mean
   is biased toward high-SD patients.  Weighting each observation by
   1/SD² (inverse-variance weighting) gives a more principled estimate.

5. MIXED-EFFECTS / RELIABILITY MODELS
   In proper modelling the SD feeds into the within-subject variance
   component.  Without it you cannot distinguish:
       • true inter-patient differences in periodicity
       • from measurement unreliability

RECOMMENDATION:
   → Always keep (mean, SD) as two separate numeric columns (done above).
   → Run correlations for acf_peak_sd_nm and acf_peak_cv in addition
     to acf_peak_mean_nm (see Section 8 below).
   → Consider a weighted Spearman or a linear model with 1/SD² weights.
=============================================================
""")

# ── 8. Correlation: ACF peak SD and CV vs. clinical variables ────────────────
print("=== Spearman: acf_peak_sd_nm  vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_sd_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_sd_nm"], sub[clin])
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

# ── 9. Heatmap of Spearman rho ───────────────────────────────────────────────
for sheet_name, df in frames.items():
    cols_for_heatmap = CLINICAL + ["acf_peak_mean_nm", "acf_peak_sd_nm", "acf_peak_cv"]
    sub = df[[c for c in cols_for_heatmap if c in df.columns]].dropna(how="all")
    if sub.shape[0] < 5:
        continue
    corr_matrix = sub.corr(method="spearman", numeric_only=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        corr_matrix,
        annot=True, fmt=".2f", center=0,
        cmap="RdBu_r", vmin=-1, vmax=1,
        linewidths=0.5, ax=ax
    )
    ax.set_title(f"Spearman ρ matrix — {sheet_name}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"corr_heatmap_{sheet_name}.png", dpi=150)
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}.png")

# ── 10. Export harmonised table ───────────────────────────────────────────────
with pd.ExcelWriter("acf_harmonised.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in frames.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined.to_excel(writer, sheet_name="combined", index=False)

print("\nHarmonised file written: acf_harmonised.xlsx")
print("* (p<0.05)   † (p<0.10)")

In [ ]:
"""
ACF Periodicity — Detailed Correlation Scatter Plots
=====================================================
Produces publication-ready 2D scatter plots for the four
most significant ACF peak vs. clinical variable correlations.

Each panel shows:
  • Raw data points coloured by group
  • Regression line + 95 % CI band (via scipy linregress)
  • Spearman ρ and p-value annotation
  • Patient ID labels (offset to avoid overlap)

Output
  • acf_scatter_panels.png  — all 4 in one figure  (2×2)
  • acf_scatter_<pair>.png  — individual high-res PNGs
"""

import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

# ── palette & style ──────────────────────────────────────────────────────────
sns.set_theme(style="ticks", font_scale=1.1)

GROUP_COLORS = {
    0.0: "#4dac26",   # control   → green
    1.0: "#d01c8b",   # diabetic neuropathy → magenta
    2.0: "#f1b6da",   # diabetes w/o neuropathy → light pink
}
GROUP_LABELS = {
    0.0: "Control",
    1.0: "Diabetic neuropathy",
    2.0: "Diabetes w/o neuropathy",
}

# ── reload & harmonise data (self-contained cell) ────────────────────────────
FILE = "../data/metadata/Update_list_for_correlation_Johannes.xlsx"

COL_MAP = {
    "Patients ":                   "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":               "diabetes_type",
    "age":                         "age",
    "sex":                         "sex",
    "Duration of Disease (months)":"disease_duration_months",
    "SNAP (sural nerve)":          "SNAP_sural",
    "HbA1c (%)":                   "HbA1c_pct",
    "NIS-LL":                      "NIS_LL",
    "SAS":                         "SAS",
    "NPSI":                        "NPSI",
    "IENFD(per mm)":               "IENFD_per_mm",
}

_PAT = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)\s*\(\s*[±+\-]?\s*([0-9]+(?:[.,][0-9]+)?)\s*\)\s*$"
)

def fix_decimal(v):
    if pd.isna(v): return np.nan
    try: return float(str(v).strip().replace(",", "."))
    except ValueError: return np.nan

def parse_mean_sd(cell):
    if pd.isna(cell): return np.nan, np.nan
    m = _PAT.match(str(cell).strip())
    if m:
        return float(m.group(1).replace(",",".")), float(m.group(2).replace(",","."))
    try: return float(str(cell).replace(",",".")), np.nan
    except ValueError: return np.nan, np.nan

PERIODICITY_RAW = {"Caspr1": "Caspr1 periodicity , mean (±SD) ", "PanNF": "PanNF"}
NUMERIC_COLS = ["age","disease_duration_months","SNAP_sural",
                "HbA1c_pct","NIS_LL","SAS","NPSI","IENFD_per_mm"]

sheets_raw = pd.read_excel(FILE, sheet_name=None)
frames = {}
for sname, df in sheets_raw.items():
    if sname not in PERIODICITY_RAW: continue
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].rename(columns=COL_MAP)
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)
    for col in NUMERIC_COLS:
        if col in df.columns: df[col] = df[col].apply(fix_decimal)
    raw_col = PERIODICITY_RAW[sname]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"] = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]   = [p[1] for p in parsed]
        df = df.drop(columns=[raw_col])
    frames[sname] = df

# ── helper: one scatter panel ────────────────────────────────────────────────
def scatter_panel(ax, df, x_col, y_col, title,
                  x_label, y_label, show_ids=True):
    """Draw one correlation scatter with regression + CI + annotations."""

    sub = df[[x_col, y_col, "group", "patient_id"]].dropna()
    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)

    # --- Spearman ---
    rho, pval = stats.spearmanr(x, y)
    p_str = f"p = {pval:.4f}" if pval >= 0.0001 else "p < 0.0001"
    annot = f"Spearman ρ = {rho:+.3f}\n{p_str}  (n = {len(x)})"

    # --- OLS regression line + 95 % CI (for visual trend only) ---
    slope, intercept, _, _, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = slope * x_line + intercept

    # bootstrap CI for the regression line
    n_boot = 2000
    rng = np.random.default_rng(42)
    boot_lines = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        s, i, *_ = stats.linregress(x[idx], y[idx])
        boot_lines.append(s * x_line + i)
    ci_lo = np.percentile(boot_lines, 2.5, axis=0)
    ci_hi = np.percentile(boot_lines, 97.5, axis=0)

    ax.fill_between(x_line, ci_lo, ci_hi, alpha=0.15, color="#555555", label="95 % CI")
    ax.plot(x_line, y_line, color="#333333", lw=1.8, ls="--", label="OLS trend")

    # --- scatter points coloured by group ---
    for grp_val, grp_df in sub.groupby("group"):
        c = GROUP_COLORS.get(grp_val, "#999999")
        ax.scatter(grp_df[x_col], grp_df[y_col],
                   color=c, s=60, edgecolors="white", linewidths=0.6,
                   zorder=3, label=GROUP_LABELS.get(grp_val, f"Group {grp_val}"))

    # patient ID annotations intentionally omitted

    # --- stat annotation box ---
    ax.text(0.97, 0.05, annot,
            transform=ax.transAxes,
            ha="right", va="bottom", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#cccccc", alpha=0.85))

    ax.set_xlabel(x_label, fontsize=10)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    sns.despine(ax=ax)
    return rho, pval

# ── define the four panels ───────────────────────────────────────────────────
PANELS = [
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="NPSI",
         title="Caspr1 — ACF peak mean vs. NPSI",
         x_label="Caspr1 ACF peak mean (nm)", y_label="NPSI score"),
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="SAS",
         title="Caspr1 — ACF peak mean vs. SAS",
         x_label="Caspr1 ACF peak mean (nm)", y_label="SAS score"),
    dict(sheet="PanNF",  x="acf_peak_sd_nm",   y="NIS_LL",
         title="PanNF — ACF peak SD vs. NIS-LL",
         x_label="PanNF ACF peak SD (nm)", y_label="NIS-LL score"),
    dict(sheet="PanNF",  x="acf_peak_mean_nm", y="NIS_LL",
         title="PanNF — ACF peak mean vs. NIS-LL",
         x_label="PanNF ACF peak mean (nm)", y_label="NIS-LL score"),
]

# ── build shared legend handles ──────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=c, label=l) for c, l in
    zip(GROUP_COLORS.values(), GROUP_LABELS.values())
] + [
    Line2D([0],[0], color="#333333", lw=1.8, ls="--", label="OLS trend"),
    mpatches.Patch(color="#555555", alpha=0.25, label="95 % CI (bootstrap)"),
]

# ── individual high-res PNGs ─────────────────────────────────────────────────
for p in PANELS:
    fig_i, ax_i = plt.subplots(figsize=(6, 5))
    scatter_panel(ax_i, frames[p["sheet"]],
                  p["x"], p["y"], p["title"],
                  p["x_label"], p["y_label"])
    ax_i.legend(handles=legend_handles, fontsize=8,
                loc="upper left", frameon=True)
    plt.tight_layout()
    fname = f"acf_scatter_{p['sheet']}_{p['x']}_vs_{p['y']}.png"
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
"""
ACF Periodicity — Stratified Correlation Scatter Plots
=======================================================
For each ACF measure (peak mean nm, peak SD nm) × clinical variable pair,
produces TWO side-by-side panels per PNG:

  LEFT  — Group 1 only  (diabetic neuropathy)
  RIGHT — Group 1 + 2   (all diabetic patients, sensitivity check)

Controls (Group 0) are intentionally excluded from correlations:
  • no meaningful clinical scores (NIS-LL, NPSI, SAS ≈ 0 by definition)
  • healthy people are not systematically screened → selection bias

Each panel shows:
  • Data points coloured by group
  • OLS trend line + bootstrapped 95 % CI band
  • Spearman ρ, p-value, n in annotation box

ACF measures tested as x-variable:
  • acf_peak_mean_nm  — mean periodicity distance
  • acf_peak_sd_nm    — SD of periodicity distance (structural irregularity)

Output: one PNG per (sheet × clinical_variable) combination
"""

import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

# ── style ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="ticks", font_scale=1.05)

GROUP_COLORS = {
    1.0: "#d01c8b",   # diabetic neuropathy     → magenta
    2.0: "#4575b4",   # diabetes w/o neuropathy → blue
}
GROUP_LABELS = {
    1.0: "Diabetic neuropathy (Gr.1)",
    2.0: "Diabetes w/o neuropathy (Gr.2)",
}

# ── file path — adjust if needed ──────────────────────────────────────────────
FILE = "../data/metadata/Update_list_for_correlation_Johannes.xlsx"

# ── column harmonisation ──────────────────────────────────────────────────────
COL_MAP = {
    "Patients ":                   "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":               "diabetes_type",
    "age":                         "age",
    "sex":                         "sex",
    "Duration of Disease (months)":"disease_duration_months",
    "SNAP (sural nerve)":          "SNAP_sural",
    "HbA1c (%)":                   "HbA1c_pct",
    "NIS-LL":                      "NIS_LL",
    "SAS":                         "SAS",
    "NPSI":                        "NPSI",
    "IENFD(per mm)":               "IENFD_per_mm",
}

_PAT = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)\s*\(\s*[±+\-]?\s*([0-9]+(?:[.,][0-9]+)?)\s*\)\s*$"
)

def fix_decimal(v):
    if pd.isna(v): return np.nan
    try: return float(str(v).strip().replace(",", "."))
    except ValueError: return np.nan

def parse_mean_sd(cell):
    if pd.isna(cell): return np.nan, np.nan
    m = _PAT.match(str(cell).strip())
    if m:
        return float(m.group(1).replace(",",".")), float(m.group(2).replace(",","."))
    try: return float(str(cell).replace(",",".")), np.nan
    except ValueError: return np.nan, np.nan

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}
NUMERIC_COLS = ["age","disease_duration_months","SNAP_sural",
                "HbA1c_pct","NIS_LL","SAS","NPSI","IENFD_per_mm"]

# ── load & harmonise ──────────────────────────────────────────────────────────
sheets_raw = pd.read_excel(FILE, sheet_name=None)
frames = {}
for sname, df in sheets_raw.items():
    if sname not in PERIODICITY_RAW: continue
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].rename(columns=COL_MAP)
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)
    for col in NUMERIC_COLS:
        if col in df.columns: df[col] = df[col].apply(fix_decimal)
    raw_col = PERIODICITY_RAW[sname]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"] = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]   = [p[1] for p in parsed]
        df["acf_peak_cv"]      = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]
        df = df.drop(columns=[raw_col])
    # exclude controls
    frames[sname] = df[df["group"].isin([1.0, 2.0])].copy()

# ── core plot helper ──────────────────────────────────────────────────────────
def draw_panel(ax, df, x_col, y_col, subtitle):
    sub = df[[x_col, y_col, "group"]].dropna()
    if len(sub) < 4:
        ax.text(0.5, 0.5, "n too small", transform=ax.transAxes,
                ha="center", va="center", color="grey")
        ax.set_title(subtitle, fontsize=10)
        return

    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)

    rho, pval = stats.spearmanr(x, y)
    p_str = f"p = {pval:.4f}" if pval >= 0.0001 else "p < 0.0001"
    sig   = "  *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
    annot = f"ρ = {rho:+.3f}{sig}\n{p_str}  (n={len(x)})"

    slope, intercept, *_ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = slope * x_line + intercept
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(2000):
        idx = rng.integers(0, len(x), len(x))
        if len(np.unique(x[idx])) < 2: continue
        s, i, *_ = stats.linregress(x[idx], y[idx])
        boots.append(s * x_line + i)
    if boots:
        ci_lo = np.percentile(boots, 2.5,  axis=0)
        ci_hi = np.percentile(boots, 97.5, axis=0)
        ax.fill_between(x_line, ci_lo, ci_hi, alpha=0.15, color="#555555", zorder=1)
    ax.plot(x_line, y_line, color="#333333", lw=1.6, ls="--", zorder=2)

    for grp_val, grp_df in sub.groupby("group"):
        c = GROUP_COLORS.get(grp_val, "#999999")
        ax.scatter(grp_df[x_col], grp_df[y_col],
                   color=c, s=55, edgecolors="white", linewidths=0.5,
                   zorder=3,
                   label=GROUP_LABELS.get(grp_val, f"Group {int(grp_val)}"))

    box_color = "#fff0f5" if pval < 0.05 else "white"
    ax.text(0.97, 0.05, annot,
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.4", fc=box_color, ec="#cccccc", alpha=0.9))

    ax.set_title(subtitle, fontsize=10, fontweight="bold")
    sns.despine(ax=ax)

# ── pairs to plot ─────────────────────────────────────────────────────────────
PAIRS = [
    # original significant results
    ("Caspr1", "acf_peak_mean_nm", "NPSI",   "Caspr1 ACF peak mean (nm)", "NPSI score"),
    ("Caspr1", "acf_peak_mean_nm", "SAS",    "Caspr1 ACF peak mean (nm)", "SAS score"),
    ("PanNF",  "acf_peak_mean_nm", "NIS_LL", "PanNF ACF peak mean (nm)",  "NIS-LL score"),
    # SD as predictor
    ("Caspr1", "acf_peak_sd_nm",   "NPSI",   "Caspr1 ACF peak SD (nm)",   "NPSI score"),
    ("Caspr1", "acf_peak_sd_nm",   "SAS",    "Caspr1 ACF peak SD (nm)",   "SAS score"),
    ("PanNF",  "acf_peak_sd_nm",   "NIS_LL", "PanNF ACF peak SD (nm)",    "NIS-LL score"),
    ("PanNF",  "acf_peak_sd_nm",   "NPSI",   "PanNF ACF peak SD (nm)",    "NPSI score"),
    # additional
    ("Caspr1", "acf_peak_mean_nm", "NIS_LL", "Caspr1 ACF peak mean (nm)", "NIS-LL score"),
    ("Caspr1", "acf_peak_sd_nm",   "NIS_LL", "Caspr1 ACF peak SD (nm)",   "NIS-LL score"),
    ("PanNF",  "acf_peak_mean_nm", "NPSI",   "PanNF ACF peak mean (nm)",  "NPSI score"),
]

legend_handles = [
    mpatches.Patch(color=c, label=l)
    for c, l in zip(GROUP_COLORS.values(), GROUP_LABELS.values())
] + [
    Line2D([0],[0], color="#333333", lw=1.6, ls="--", label="OLS trend"),
    mpatches.Patch(color="#555555", alpha=0.2, label="95 % CI (bootstrap)"),
]

# ── generate one PNG per pair ─────────────────────────────────────────────────
for (sheet, x_col, y_col, x_label, y_label) in PAIRS:
    df_all = frames[sheet]
    df_gr1 = df_all[df_all["group"] == 1.0]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=False)
    fig.suptitle(f"{sheet}  |  {x_label}  vs.  {y_label}",
                 fontsize=12, fontweight="bold")

    draw_panel(axes[0], df_gr1, x_col, y_col,
               "Group 1 only — Diabetic neuropathy")
    axes[0].set_xlabel(x_label, fontsize=9)
    axes[0].set_ylabel(y_label, fontsize=9)

    draw_panel(axes[1], df_all, x_col, y_col,
               "Group 1 + 2 — All diabetic patients (sensitivity)")
    axes[1].set_xlabel(x_label, fontsize=9)
    axes[1].set_ylabel(y_label, fontsize=9)

    fig.legend(handles=legend_handles, loc="lower center",
               ncol=4, fontsize=8.5, frameon=True,
               bbox_to_anchor=(0.5, -0.08))
    plt.tight_layout(rect=[0, 0.05, 1, 1])

    fname = f"acf_scatter_{sheet}_{x_col}_vs_{y_col}.png"
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {fname}")

print("\nDone. Controls (Group 0) excluded from all correlations.")
print("Pink annotation box = p < 0.05  |  * p<0.05  † p<0.10")

In [ ]:
"""
ACF Periodicity — Stratified Correlation Scatter Plots
=======================================================
For each ACF measure (peak mean nm, peak SD nm) × clinical variable pair,
produces TWO side-by-side panels per PNG:

  LEFT  — Group 1 only  (diabetic neuropathy)
  RIGHT — Group 1 + 2   (all diabetic patients, sensitivity check)

Controls (Group 0) are intentionally excluded from correlations:
  • no meaningful clinical scores (NIS-LL, NPSI, SAS ≈ 0 by definition)
  • healthy people are not systematically screened → selection bias

Each panel shows:
  • Data points coloured by group
  • OLS trend line + bootstrapped 95 % CI band
  • Spearman ρ, p-value, n in annotation box

ACF measures tested as x-variable:
  • acf_peak_mean_nm  — mean periodicity distance
  • acf_peak_sd_nm    — SD of periodicity distance (structural irregularity)

Output: one PNG per (sheet × clinical_variable) combination
"""

import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

# ── style ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="ticks", font_scale=1.05)

GROUP_COLORS = {
    1.0: "#d01c8b",   # diabetic neuropathy     → magenta
    2.0: "#4575b4",   # diabetes w/o neuropathy → blue
}
GROUP_LABELS = {
    1.0: "Diabetic neuropathy (Gr.1)",
    2.0: "Diabetes w/o neuropathy (Gr.2)",
}

# ── file path — adjust if needed ──────────────────────────────────────────────
FILE = "../data/metadata/Update_list_for_correlation_Johannes.xlsx"

# ── column harmonisation ──────────────────────────────────────────────────────
COL_MAP = {
    "Patients ":                   "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":               "diabetes_type",
    "age":                         "age",
    "sex":                         "sex",
    "Duration of Disease (months)":"disease_duration_months",
    "SNAP (sural nerve)":          "SNAP_sural",
    "HbA1c (%)":                   "HbA1c_pct",
    "NIS-LL":                      "NIS_LL",
    "SAS":                         "SAS",
    "NPSI":                        "NPSI",
    "IENFD(per mm)":               "IENFD_per_mm",
}

_PAT = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)\s*\(\s*[±+\-]?\s*([0-9]+(?:[.,][0-9]+)?)\s*\)\s*$"
)

def fix_decimal(v):
    if pd.isna(v): return np.nan
    try: return float(str(v).strip().replace(",", "."))
    except ValueError: return np.nan

def parse_mean_sd(cell):
    if pd.isna(cell): return np.nan, np.nan
    m = _PAT.match(str(cell).strip())
    if m:
        return float(m.group(1).replace(",",".")), float(m.group(2).replace(",","."))
    try: return float(str(cell).replace(",",".")), np.nan
    except ValueError: return np.nan, np.nan

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}
NUMERIC_COLS = ["age","disease_duration_months","SNAP_sural",
                "HbA1c_pct","NIS_LL","SAS","NPSI","IENFD_per_mm"]

# ── load & harmonise ──────────────────────────────────────────────────────────
sheets_raw = pd.read_excel(FILE, sheet_name=None)
frames = {}
for sname, df in sheets_raw.items():
    if sname not in PERIODICITY_RAW: continue
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].rename(columns=COL_MAP)
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)
    for col in NUMERIC_COLS:
        if col in df.columns: df[col] = df[col].apply(fix_decimal)
    raw_col = PERIODICITY_RAW[sname]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"] = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]   = [p[1] for p in parsed]
        df["acf_peak_cv"]      = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]
        df = df.drop(columns=[raw_col])
    # exclude controls
    frames[sname] = df[df["group"].isin([1.0, 2.0])].copy()

# ── core plot helper ──────────────────────────────────────────────────────────
def draw_panel(ax, df, x_col, y_col, subtitle):
    sub = df[[x_col, y_col, "group"]].dropna()
    if len(sub) < 4:
        ax.text(0.5, 0.5, "n too small", transform=ax.transAxes,
                ha="center", va="center", color="grey")
        ax.set_title(subtitle, fontsize=10)
        return

    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)

    rho, pval = stats.spearmanr(x, y)
    p_str = f"p = {pval:.4f}" if pval >= 0.0001 else "p < 0.0001"
    sig   = "  *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
    annot = f"ρ = {rho:+.3f}{sig}\n{p_str}  (n={len(x)})"

    slope, intercept, *_ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = slope * x_line + intercept
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(2000):
        idx = rng.integers(0, len(x), len(x))
        if len(np.unique(x[idx])) < 2: continue
        s, i, *_ = stats.linregress(x[idx], y[idx])
        boots.append(s * x_line + i)
    if boots:
        ci_lo = np.percentile(boots, 2.5,  axis=0)
        ci_hi = np.percentile(boots, 97.5, axis=0)
        ax.fill_between(x_line, ci_lo, ci_hi, alpha=0.15, color="#555555", zorder=1)
    ax.plot(x_line, y_line, color="#333333", lw=1.6, ls="--", zorder=2)

    for grp_val, grp_df in sub.groupby("group"):
        c = GROUP_COLORS.get(grp_val, "#999999")
        ax.scatter(grp_df[x_col], grp_df[y_col],
                   color=c, s=55, edgecolors="white", linewidths=0.5,
                   zorder=3,
                   label=GROUP_LABELS.get(grp_val, f"Group {int(grp_val)}"))

    box_color = "#fff0f5" if pval < 0.05 else "white"
    ax.text(0.97, 0.05, annot,
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.4", fc=box_color, ec="#cccccc", alpha=0.9))

    ax.set_title(subtitle, fontsize=10, fontweight="bold")
    sns.despine(ax=ax)

# ── pairs to plot ─────────────────────────────────────────────────────────────
PAIRS = [
    # original significant results
    ("Caspr1", "acf_peak_mean_nm", "NPSI",   "Caspr1 ACF peak mean (nm)", "NPSI score"),
    ("Caspr1", "acf_peak_mean_nm", "SAS",    "Caspr1 ACF peak mean (nm)", "SAS score"),
    ("PanNF",  "acf_peak_mean_nm", "NIS_LL", "PanNF ACF peak mean (nm)",  "NIS-LL score"),
    # SD as predictor
    ("Caspr1", "acf_peak_sd_nm",   "NPSI",   "Caspr1 ACF peak SD (nm)",   "NPSI score"),
    ("Caspr1", "acf_peak_sd_nm",   "SAS",    "Caspr1 ACF peak SD (nm)",   "SAS score"),
    ("PanNF",  "acf_peak_sd_nm",   "NIS_LL", "PanNF ACF peak SD (nm)",    "NIS-LL score"),
    ("PanNF",  "acf_peak_sd_nm",   "NPSI",   "PanNF ACF peak SD (nm)",    "NPSI score"),
    # additional
    ("Caspr1", "acf_peak_mean_nm", "NIS_LL", "Caspr1 ACF peak mean (nm)", "NIS-LL score"),
    ("Caspr1", "acf_peak_sd_nm",   "NIS_LL", "Caspr1 ACF peak SD (nm)",   "NIS-LL score"),
    ("PanNF",  "acf_peak_mean_nm", "NPSI",   "PanNF ACF peak mean (nm)",  "NPSI score"),
]

legend_handles = [
    mpatches.Patch(color=c, label=l)
    for c, l in zip(GROUP_COLORS.values(), GROUP_LABELS.values())
] + [
    Line2D([0],[0], color="#333333", lw=1.6, ls="--", label="OLS trend"),
    mpatches.Patch(color="#555555", alpha=0.2, label="95 % CI (bootstrap)"),
]

# ── generate one PNG per pair ─────────────────────────────────────────────────
for (sheet, x_col, y_col, x_label, y_label) in PAIRS:
    df_all = frames[sheet]
    df_gr1 = df_all[df_all["group"] == 1.0]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=False)
    fig.suptitle(f"{sheet}  |  {x_label}  vs.  {y_label}",
                 fontsize=12, fontweight="bold")

    draw_panel(axes[0], df_gr1, x_col, y_col,
               "Group 1 only — Diabetic neuropathy")
    axes[0].set_xlabel(x_label, fontsize=9)
    axes[0].set_ylabel(y_label, fontsize=9)

    draw_panel(axes[1], df_all, x_col, y_col,
               "Group 1 + 2 — All diabetic patients (sensitivity)")
    axes[1].set_xlabel(x_label, fontsize=9)
    axes[1].set_ylabel(y_label, fontsize=9)

    fig.legend(handles=legend_handles, loc="lower center",
               ncol=4, fontsize=8.5, frameon=True,
               bbox_to_anchor=(0.5, -0.08))
    plt.tight_layout(rect=[0, 0.05, 1, 1])

    fname = f"acf_scatter_{sheet}_{x_col}_vs_{y_col}.png"
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()      # renders inline in JupyterHub
    plt.close()
    print(f"Saved: {fname}")

print("\nDone. Controls (Group 0) excluded from all correlations.")
print("Pink annotation box = p < 0.05  |  * p<0.05  † p<0.10")

In [ ]:


# ── 2. Helper: fix European decimal commas and parse to float ────────────────
def fix_decimal(val):
    """Replace comma-decimal (3,1 → 3.1) and convert to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# ── 3. Helper: parse "330.02(±160)" → (mean, sd) ────────────────────────────
_PATTERN = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)"   # mean part
    r"\s*\(\s*[±+\-]?\s*"            # opening paren + optional ± sign
    r"([0-9]+(?:[.,][0-9]+)?)"       # SD part
    r"\s*\)\s*$"                     # closing paren
)

def parse_mean_sd(cell):
    """
    Returns (mean_float, sd_float) from strings like:
        '330.02(±160)'   '314.86 (±156)'   '456(±153)'
    Returns (NaN, NaN) if parsing fails.
    """
    if pd.isna(cell):
        return np.nan, np.nan
    m = _PATTERN.match(str(cell).strip())
    if m:
        mean = float(m.group(1).replace(",", "."))
        sd   = float(m.group(2).replace(",", "."))
        return mean, sd
    # fallback: try plain number (no SD given)
    try:
        return float(str(cell).replace(",", ".")), np.nan
    except ValueError:
        return np.nan, np.nan

# ── 4. Process each sheet ────────────────────────────────────────────────────
NUMERIC_COLS = [
    "age", "disease_duration_months", "SNAP_sural",
    "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm",
]

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}

frames = {}
for sheet_name, df in sheets_raw.items():
    if sheet_name not in PERIODICITY_RAW:
        continue

    # drop unnamed helper columns
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # rename known columns
    df = df.rename(columns=COL_MAP)

    # fix group column: strip text annotations like "1 (SFN)"
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)

    # fix sex: binary encode
    df["sex_male"] = (df["sex"].astype(str).str.lower().str.strip() == "male").astype(int)

    # fix numeric columns with possible comma-decimals
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = df[col].apply(fix_decimal)

    # parse periodicity mean ± SD
    raw_col = PERIODICITY_RAW[sheet_name]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"]  = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]    = [p[1] for p in parsed]
        df["acf_peak_cv"]       = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]  # coefficient of variation
        df = df.drop(columns=[raw_col])

    frames[sheet_name] = df
    print(f"\n=== {sheet_name}: {df.shape[0]} rows, {df.shape[1]} cols ===")
    print(df[["patient_id", "group", "age", "acf_peak_mean_nm", "acf_peak_sd_nm"]].to_string(index=False))

# ── 5. Combine for joint analysis ────────────────────────────────────────────
for name, df in frames.items():
    df["marker"] = name

combined = pd.concat(frames.values(), ignore_index=True)

# ── 6. Spearman correlation: ACF peak mean vs. clinical variables ────────────
CLINICAL = ["age", "disease_duration_months", "SNAP_sural",
            "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm"]

print("\n\n=== Spearman correlations: acf_peak_mean_nm vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)

results = []
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_mean_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_mean_nm"], sub[clin])
        results.append(dict(
            marker=sheet_name, variable=clin,
            rho=rho, pval=pval, n=len(sub)
        ))
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

results_df = pd.DataFrame(results)

# ── 7. Why SD/variance of the ACF peak matters ───────────────────────────────
print("""
=============================================================
WHY VARIANCE (SD) OF THE ACF PEAK DISTANCE MATTERS
=============================================================

1. BIOLOGICAL NOISE vs. TRUE SIGNAL
   The SD you already stored reflects within-axon variability of
   node-of-Ranvier spacing.  A large SD means the periodicity is
   irregular — pathological remodelling, demyelination, or
   measurement noise.  Using only the mean ignores this.

2. CORRELATION POWER
   If two patients share the same mean periodicity but very different
   SDs, their nerve physiology differs.  Regressing mean alone against,
   e.g., NIS-LL will leave variance-driven variance unexplained,
   inflating residuals and reducing statistical power.

3. SEPARATE PREDICTORS
   acf_peak_sd_nm (or CV = SD/mean) is itself a clinical correlate.
   It may correlate with disease severity (NIS-LL, NPSI) independently
   of the mean, because irregular spacing disrupts saltatory conduction
   even when average spacing is normal.

4. WEIGHTED CORRELATIONS
   If SD differs greatly across patients, Pearson/Spearman on the mean
   is biased toward high-SD patients.  Weighting each observation by
   1/SD² (inverse-variance weighting) gives a more principled estimate.

5. MIXED-EFFECTS / RELIABILITY MODELS
   In proper modelling the SD feeds into the within-subject variance
   component.  Without it you cannot distinguish:
       • true inter-patient differences in periodicity
       • from measurement unreliability

RECOMMENDATION:
   → Always keep (mean, SD) as two separate numeric columns (done above).
   → Run correlations for acf_peak_sd_nm and acf_peak_cv in addition
     to acf_peak_mean_nm (see Section 8 below).
   → Consider a weighted Spearman or a linear model with 1/SD² weights.
=============================================================
""")

# ── 8. Correlation: ACF peak SD and CV vs. clinical variables ────────────────
print("=== Spearman: acf_peak_sd_nm  vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_sd_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_sd_nm"], sub[clin])
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

# ── 9. Heatmap of Spearman rho — Group 1+2 combined, reliable pairs highlighted
#
# Rationale for using Group 1+2:
#   Periodicity degradation drives clinical scores (pain, motor/sensory deficit).
#   Diabetes is a risk factor for that degradation — not the direct cause.
#   Therefore all diabetic patients (with AND without neuropathy) form the
#   appropriate continuous population for correlating ACF measures with
#   clinical outcomes.  Controls are excluded (no meaningful clinical scores,
#   selection bias).
#
# "Reliable" = p < 0.05 in Group 1+2 AND biologically plausible direction.
#   Caspr1 mean  ↔ NPSI   (ρ=+0.66*)
#   Caspr1 mean  ↔ SAS    (ρ=+0.55*)
#   PanNF  SD    ↔ NIS-LL (ρ=+0.60*)
# ─────────────────────────────────────────────────────────────────────────────

# Reliable pairs per sheet: {sheet: [(row_var, col_var), ...]}
RELIABLE = {
    "Caspr1": [
        ("acf_peak_mean_nm", "NPSI"),
        ("NPSI", "acf_peak_mean_nm"),
        ("acf_peak_mean_nm", "SAS"),
        ("SAS", "acf_peak_mean_nm"),
    ],
    "PanNF": [
        ("acf_peak_sd_nm", "NIS_LL"),
        ("NIS_LL", "acf_peak_sd_nm"),
    ],
}

for sheet_name, df in frames.items():
    cols_for_heatmap = CLINICAL + ["acf_peak_mean_nm", "acf_peak_sd_nm", "acf_peak_cv"]

    # use Group 1+2 combined (exclude controls)
    sub = df[df["group"].isin([1.0, 2.0])][[
        c for c in cols_for_heatmap if c in df.columns
    ]].dropna(how="all")
    if sub.shape[0] < 5:
        continue

    corr_matrix = sub.corr(method="spearman", numeric_only=True)

    # compute p-value matrix for significance markers
    cols = corr_matrix.columns.tolist()
    pval_matrix = pd.DataFrame(np.ones((len(cols), len(cols))), index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            if i != j:
                pair = sub[[c1, c2]].dropna()
                if len(pair) >= 5:
                    _, p = stats.spearmanr(pair[c1], pair[c2])
                    pval_matrix.loc[c1, c2] = p

    # build annotation: ρ value + significance star
    annot_matrix = pd.DataFrame("", index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            r = corr_matrix.loc[c1, c2]
            p = pval_matrix.loc[c1, c2]
            if i == j:
                annot_matrix.loc[c1, c2] = "1.00"
            else:
                star = "*" if p < 0.05 else ("†" if p < 0.10 else "")
                annot_matrix.loc[c1, c2] = f"{r:.2f}{star}"

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        corr_matrix,
        annot=annot_matrix, fmt="", center=0,
        cmap="RdBu_r", vmin=-1, vmax=1,
        linewidths=0.5, ax=ax,
        annot_kws={"size": 8}
    )

    # overlay thick green border on reliable cells
    reliable_pairs = RELIABLE.get(sheet_name, [])
    for (row_var, col_var) in reliable_pairs:
        if row_var in cols and col_var in cols:
            r_idx = cols.index(row_var)
            c_idx = cols.index(col_var)
            ax.add_patch(plt.Rectangle(
                (c_idx, r_idx), 1, 1,
                fill=False, edgecolor="#00aa00", lw=3, zorder=5
            ))

    # legend note
    ax.text(1.02, 0.5,
            "Green border = reliable\ncorrelation (p<0.05,\nGroup 1+2 combined)\n\n"
            "* p<0.05\n† p<0.10",
            transform=ax.transAxes, fontsize=8.5, va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="#cccccc"))

    ax.set_title(
        f"Spearman ρ matrix — {sheet_name}  (Group 1+2, controls excluded)",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0, 0.88, 1])
    plt.savefig(f"corr_heatmap_{sheet_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}.png")

# ── 10. Export harmonised table ───────────────────────────────────────────────
with pd.ExcelWriter("acf_harmonised.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in frames.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined.to_excel(writer, sheet_name="combined", index=False)

print("\nHarmonised file written: acf_harmonised.xlsx")
print("* (p<0.05)   † (p<0.10)")

In [ ]:
"""
ACF Periodicity Correlation Analysis
=====================================
Harmonizes the clinical data table (Caspr1 & PanNF sheets),
parses mean ± SD from embedded strings, and runs
Spearman correlations between ACF periodicity distances
and clinical variables.

Usage:
    python acf_correlation_analysis.py
    # or paste cells into a Jupyter notebook
"""

import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# ── 0. Load ──────────────────────────────────────────────────────────────────
FILE = "../data/metadata/acf_harmonised.xlsx"            # adjust path if needed

sheets_raw = pd.read_excel(FILE, sheet_name=None)

# ── 1. Column name harmonisation map ────────────────────────────────────────
COL_MAP = {
    "Patients ":                                                       "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":                                                   "diabetes_type",
    "age":                                                             "age",
    "sex":                                                             "sex",
    "Duration of Disease (months)":                                    "disease_duration_months",
    "SNAP (sural nerve)":                                              "SNAP_sural",
    "HbA1c (%)":                                                       "HbA1c_pct",
    "NIS-LL":                                                          "NIS_LL",
    "SAS":                                                             "SAS",
    "NPSI":                                                            "NPSI",
    "IENFD(per mm)":                                                   "IENFD_per_mm",
    # sheet-specific periodicity columns  (handled separately below)
}

# ── 2. Helper: fix European decimal commas and parse to float ────────────────
def fix_decimal(val):
    """Replace comma-decimal (3,1 → 3.1) and convert to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# ── 3. Helper: parse "330.02(±160)" → (mean, sd) ────────────────────────────
_PATTERN = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)"   # mean part
    r"\s*\(\s*[±+\-]?\s*"            # opening paren + optional ± sign
    r"([0-9]+(?:[.,][0-9]+)?)"       # SD part
    r"\s*\)\s*$"                     # closing paren
)

def parse_mean_sd(cell):
    """
    Returns (mean_float, sd_float) from strings like:
        '330.02(±160)'   '314.86 (±156)'   '456(±153)'
    Returns (NaN, NaN) if parsing fails.
    """
    if pd.isna(cell):
        return np.nan, np.nan
    m = _PATTERN.match(str(cell).strip())
    if m:
        mean = float(m.group(1).replace(",", "."))
        sd   = float(m.group(2).replace(",", "."))
        return mean, sd
    # fallback: try plain number (no SD given)
    try:
        return float(str(cell).replace(",", ".")), np.nan
    except ValueError:
        return np.nan, np.nan

# ── 4. Process each sheet ────────────────────────────────────────────────────
NUMERIC_COLS = [
    "age", "disease_duration_months", "SNAP_sural",
    "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm",
]

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}

frames = {}
for sheet_name, df in sheets_raw.items():
    if sheet_name not in PERIODICITY_RAW:
        continue

    # drop unnamed helper columns
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # rename known columns
    df = df.rename(columns=COL_MAP)

    # fix group column: strip text annotations like "1 (SFN)"
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)

    # fix sex: binary encode
    df["sex_male"] = (df["sex"].astype(str).str.lower().str.strip() == "male").astype(int)

    # fix numeric columns with possible comma-decimals
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = df[col].apply(fix_decimal)

    # parse periodicity mean ± SD
    raw_col = PERIODICITY_RAW[sheet_name]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"]  = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]    = [p[1] for p in parsed]
        df["acf_peak_cv"]       = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]  # coefficient of variation
        df = df.drop(columns=[raw_col])

    frames[sheet_name] = df
    print(f"\n=== {sheet_name}: {df.shape[0]} rows, {df.shape[1]} cols ===")
    print(df[["patient_id", "group", "age", "acf_peak_mean_nm", "acf_peak_sd_nm"]].to_string(index=False))

# ── 5. Combine for joint analysis ────────────────────────────────────────────
for name, df in frames.items():
    df["marker"] = name

combined = pd.concat(frames.values(), ignore_index=True)

# ── 6. Spearman correlation: ACF peak mean vs. clinical variables ────────────
CLINICAL = ["age", "disease_duration_months", "SNAP_sural",
            "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm"]

print("\n\n=== Spearman correlations: acf_peak_mean_nm vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)

results = []
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_mean_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_mean_nm"], sub[clin])
        results.append(dict(
            marker=sheet_name, variable=clin,
            rho=rho, pval=pval, n=len(sub)
        ))
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

results_df = pd.DataFrame(results)

# ── 7. Why SD/variance of the ACF peak matters ───────────────────────────────
print("""
=============================================================
WHY VARIANCE (SD) OF THE ACF PEAK DISTANCE MATTERS
=============================================================

1. BIOLOGICAL NOISE vs. TRUE SIGNAL
   The SD you already stored reflects within-axon variability of
   node-of-Ranvier spacing.  A large SD means the periodicity is
   irregular — pathological remodelling, demyelination, or
   measurement noise.  Using only the mean ignores this.

2. CORRELATION POWER
   If two patients share the same mean periodicity but very different
   SDs, their nerve physiology differs.  Regressing mean alone against,
   e.g., NIS-LL will leave variance-driven variance unexplained,
   inflating residuals and reducing statistical power.

3. SEPARATE PREDICTORS
   acf_peak_sd_nm (or CV = SD/mean) is itself a clinical correlate.
   It may correlate with disease severity (NIS-LL, NPSI) independently
   of the mean, because irregular spacing disrupts saltatory conduction
   even when average spacing is normal.

4. WEIGHTED CORRELATIONS
   If SD differs greatly across patients, Pearson/Spearman on the mean
   is biased toward high-SD patients.  Weighting each observation by
   1/SD² (inverse-variance weighting) gives a more principled estimate.

5. MIXED-EFFECTS / RELIABILITY MODELS
   In proper modelling the SD feeds into the within-subject variance
   component.  Without it you cannot distinguish:
       • true inter-patient differences in periodicity
       • from measurement unreliability

RECOMMENDATION:
   → Always keep (mean, SD) as two separate numeric columns (done above).
   → Run correlations for acf_peak_sd_nm and acf_peak_cv in addition
     to acf_peak_mean_nm (see Section 8 below).
   → Consider a weighted Spearman or a linear model with 1/SD² weights.
=============================================================
""")

# ── 8. Correlation: ACF peak SD and CV vs. clinical variables ────────────────
print("=== Spearman: acf_peak_sd_nm  vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_sd_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_sd_nm"], sub[clin])
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

# ── 9. Heatmap of Spearman rho — Group 1+2 combined, reliable pairs highlighted
# ─────────────────────────────────────────────────────────────────────────────
# Rationale for Group 1+2:
#   Periodicity degradation (not diabetes per se) drives clinical scores.
#   Diabetes is a risk factor for that degradation → all diabetic patients
#   form the appropriate continuous population.  Controls excluded (no
#   meaningful clinical scores, selection bias).
#
# Reliable = p < 0.05 in Group 1+2, biologically plausible direction:
#   Caspr1 mean  ↔ NPSI   (ρ=+0.66*)
#   Caspr1 mean  ↔ SAS    (ρ=+0.55*)
#   PanNF  SD    ↔ NIS-LL (ρ=+0.60*)
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

RELIABLE = {
    "Caspr1": [
        ("acf_peak_mean_nm", "NPSI"),  ("NPSI", "acf_peak_mean_nm"),
        ("acf_peak_mean_nm", "SAS"),   ("SAS",  "acf_peak_mean_nm"),
    ],
    "PanNF": [
        ("acf_peak_sd_nm", "NIS_LL"), ("NIS_LL", "acf_peak_sd_nm"),
    ],
}

for sheet_name, df in frames.items():
    cols_for_heatmap = CLINICAL + ["acf_peak_mean_nm", "acf_peak_sd_nm", "acf_peak_cv"]
    sub = df[df["group"].isin([1.0, 2.0])][[
        c for c in cols_for_heatmap if c in df.columns
    ]].dropna(how="all")
    if sub.shape[0] < 5:
        continue

    corr_matrix = sub.corr(method="spearman", numeric_only=True)
    cols = corr_matrix.columns.tolist()

    # p-value matrix
    pval_matrix = pd.DataFrame(np.ones((len(cols), len(cols))), index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            if i != j:
                pair = sub[[c1, c2]].dropna()
                if len(pair) >= 5:
                    _, p = stats.spearmanr(pair[c1], pair[c2])
                    pval_matrix.loc[c1, c2] = p

    # annotation: rho + star, no overlap with legend box
    annot_matrix = pd.DataFrame("", index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            r = corr_matrix.loc[c1, c2]
            p = pval_matrix.loc[c1, c2]
            if i == j:
                annot_matrix.loc[c1, c2] = "1.00"
            else:
                star = "*" if p < 0.05 else ("†" if p < 0.10 else "")
                annot_matrix.loc[c1, c2] = f"{r:.2f}{star}"

    # ── heatmap figure (no inline legend box — separate file below) ───────────
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        corr_matrix,
        annot=annot_matrix, fmt="", center=0,
        cmap="RdBu_r", vmin=-1, vmax=1,
        linewidths=0.5, ax=ax,
        annot_kws={"size": 8.5},
        cbar_kws={"shrink": 0.8, "pad": 0.02},
    )

    # green borders on reliable cells
    for (row_var, col_var) in RELIABLE.get(sheet_name, []):
        if row_var in cols and col_var in cols:
            ax.add_patch(plt.Rectangle(
                (cols.index(col_var), cols.index(row_var)), 1, 1,
                fill=False, edgecolor="#00aa00", lw=3.5, zorder=5
            ))

    ax.set_title(
        f"Spearman ρ matrix — {sheet_name}  (Group 1+2, controls excluded)",
        fontsize=13, fontweight="bold", pad=14
    )
    ax.tick_params(axis="x", labelsize=9, rotation=45)
    ax.tick_params(axis="y", labelsize=9, rotation=0)
    plt.tight_layout()
    plt.savefig(f"corr_heatmap_{sheet_name}.png", dpi=200, bbox_inches="tight",
                facecolor="white")
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}.png")

    # ── separate legend PNG for this heatmap ─────────────────────────────────
    fig_leg, ax_leg = plt.subplots(figsize=(5.5, 0.9))
    ax_leg.axis("off")
    legend_items = [
        mpatches.Patch(facecolor="none", edgecolor="#00aa00", linewidth=2.5,
                       label="Reliable correlation (p<0.05, Group 1+2)"),
        mpatches.Patch(facecolor="white", edgecolor="none", label="* p<0.05   † p<0.10"),
    ]
    ax_leg.legend(handles=legend_items, loc="center", ncol=2,
                  fontsize=11, frameon=True, framealpha=1,
                  edgecolor="#cccccc", handlelength=1.5)
    fig_leg.savefig(f"corr_heatmap_{sheet_name}_legend.png", dpi=200,
                    bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}_legend.png")


# ── 10. Publication 3-panel scatter (reliable correlations only) ──────────────
# Panels: A Caspr1 mean↔NPSI | B Caspr1 mean↔SAS | C PanNF SD↔NIS-LL
# Group 1+2 combined.  No legend in main figure — saved separately.
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 14, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "axes.linewidth": 1.2, "xtick.major.width": 1.2, "ytick.major.width": 1.2,
    "xtick.major.size": 5,  "ytick.major.size": 5,
})

GROUP_COLORS  = {1.0: "#c0392b", 2.0: "#2980b9"}
GROUP_MARKERS = {1.0: "o",       2.0: "s"}
GROUP_LABELS  = {1.0: "Diabetic neuropathy (Gr. 1)",
                 2.0: "Diabetes w/o neuropathy (Gr. 2)"}

PANELS = [
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="NPSI",
         x_label="Caspr1 ACF peak mean (nm)", y_label="NPSI score",   label="A"),
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="SAS",
         x_label="Caspr1 ACF peak mean (nm)", y_label="SAS score",    label="B"),
    dict(sheet="PanNF",  x="acf_peak_sd_nm",  y="NIS_LL",
         x_label="PanNF ACF peak SD (nm)",    y_label="NIS-LL score", label="C"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.subplots_adjust(wspace=0.38)

for ax, p in zip(axes, PANELS):
    df  = frames[p["sheet"]]
    sub = df[[p["x"], p["y"], "group"]].dropna()
    x   = sub[p["x"]].values.astype(float)
    y   = sub[p["y"]].values.astype(float)

    rho, pval = stats.spearmanr(x, y)
    p_str = f"p = {pval:.4f}" if pval >= 0.001 else f"p = {pval:.2e}"
    sig   = " *" if pval < 0.05 else (" †" if pval < 0.10 else "")
    annot = f"Spearman ρ = {rho:+.3f}{sig}\n{p_str}  (n = {len(x)})"

    # OLS + bootstrap CI
    slope, intercept, *_ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 300)
    y_line = slope * x_line + intercept
    rng = np.random.default_rng(42)
    boots = [stats.linregress(x[idx := rng.integers(0, len(x), len(x))], y[idx])[:2]
             for _ in range(3000)
             if len(np.unique(x[rng.integers(0, len(x), len(x))])) > 1]
    boot_y = np.array([s * x_line + i for s, i in boots])
    ax.fill_between(x_line, np.percentile(boot_y, 2.5, axis=0),
                    np.percentile(boot_y, 97.5, axis=0),
                    alpha=0.18, color="#888888", zorder=1)
    ax.plot(x_line, y_line, color="#333333", lw=2, ls="--", zorder=2)

    for grp_val, grp_df in sub.groupby("group"):
        ax.scatter(grp_df[p["x"]], grp_df[p["y"]],
                   color=GROUP_COLORS.get(grp_val, "#aaa"),
                   marker=GROUP_MARKERS.get(grp_val, "o"),
                   s=80, edgecolors="white", linewidths=0.7, zorder=3)

    box_fc = "#fff0f0" if pval < 0.05 else "white"
    ax.text(0.97, 0.04, annot, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc=box_fc, ec="#aaaaaa", lw=1.2))

    ax.text(-0.13, 1.04, p["label"], transform=ax.transAxes,
            fontsize=18, fontweight="bold", va="top")

    ax.set_xlabel(p["x_label"], labelpad=8)
    ax.set_ylabel(p["y_label"], labelpad=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    xpad = (x.max() - x.min()) * 0.08
    ypad = (y.max() - y.min()) * 0.10
    ax.set_xlim(x.min() - xpad, x.max() + xpad)
    ax.set_ylim(y.min() - ypad, y.max() + ypad * 2.5)

fig.suptitle(
    "ACF Periodicity Correlates with Clinical Severity  (Group 1+2, controls excluded)",
    fontsize=15, fontweight="bold", y=1.03
)
plt.savefig("acf_publication_figure.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig("acf_publication_figure.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: acf_publication_figure.png / .pdf")

# ── separate scatter legend ───────────────────────────────────────────────────
scatter_legend_handles = [
    mpatches.Patch(color=GROUP_COLORS[1.0], label=GROUP_LABELS[1.0]),
    mpatches.Patch(color=GROUP_COLORS[2.0], label=GROUP_LABELS[2.0]),
    Line2D([0],[0], color="#333333", lw=2, ls="--", label="OLS regression"),
    mpatches.Patch(color="#888888", alpha=0.3, label="95 % CI (bootstrap, n=3,000)"),
]
fig_leg2, ax_leg2 = plt.subplots(figsize=(9, 0.75))
ax_leg2.axis("off")
ax_leg2.legend(handles=scatter_legend_handles, loc="center", ncol=4,
               fontsize=11, frameon=True, framealpha=1, edgecolor="#cccccc",
               handlelength=1.8, handletextpad=0.7, columnspacing=1.8)
fig_leg2.savefig("acf_publication_figure_legend.png", dpi=300,
                 bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: acf_publication_figure_legend.png  (add in Photoshop)")

# ── 11. Export harmonised table ───────────────────────────────────────────────
with pd.ExcelWriter("acf_harmonised.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in frames.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined.to_excel(writer, sheet_name="combined", index=False)

print("\nHarmonised file written: acf_harmonised.xlsx")
print("* (p<0.05)   † (p<0.10)")

In [ ]:
"""
ACF Periodicity Correlation Analysis
=====================================
Harmonizes the clinical data table (Caspr1 & PanNF sheets),
parses mean ± SD from embedded strings, and runs
Spearman correlations between ACF periodicity distances
and clinical variables.

Usage:
    python acf_correlation_analysis.py
    # or paste cells into a Jupyter notebook
"""

import pandas as pd
import numpy as np
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# ── 0. Load ──────────────────────────────────────────────────────────────────
FILE = "../data/metadata/acf_harmonised.xlsx"          # adjust path if needed

sheets_raw = pd.read_excel(FILE, sheet_name=None)

# ── 1. Column name harmonisation map ────────────────────────────────────────
COL_MAP = {
    "Patients ":                                                       "patient_id",
    "Group (1=diabetic neuropathy, 2=diabetes without neuropathy, 0=control)": "group",
    "diabetes type":                                                   "diabetes_type",
    "age":                                                             "age",
    "sex":                                                             "sex",
    "Duration of Disease (months)":                                    "disease_duration_months",
    "SNAP (sural nerve)":                                              "SNAP_sural",
    "HbA1c (%)":                                                       "HbA1c_pct",
    "NIS-LL":                                                          "NIS_LL",
    "SAS":                                                             "SAS",
    "NPSI":                                                            "NPSI",
    "IENFD(per mm)":                                                   "IENFD_per_mm",
    # sheet-specific periodicity columns  (handled separately below)
}

# ── 2. Helper: fix European decimal commas and parse to float ────────────────
def fix_decimal(val):
    """Replace comma-decimal (3,1 → 3.1) and convert to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# ── 3. Helper: parse "330.02(±160)" → (mean, sd) ────────────────────────────
_PATTERN = re.compile(
    r"^\s*([0-9]+(?:[.,][0-9]+)?)"   # mean part
    r"\s*\(\s*[±+\-]?\s*"            # opening paren + optional ± sign
    r"([0-9]+(?:[.,][0-9]+)?)"       # SD part
    r"\s*\)\s*$"                     # closing paren
)

def parse_mean_sd(cell):
    """
    Returns (mean_float, sd_float) from strings like:
        '330.02(±160)'   '314.86 (±156)'   '456(±153)'
    Returns (NaN, NaN) if parsing fails.
    """
    if pd.isna(cell):
        return np.nan, np.nan
    m = _PATTERN.match(str(cell).strip())
    if m:
        mean = float(m.group(1).replace(",", "."))
        sd   = float(m.group(2).replace(",", "."))
        return mean, sd
    # fallback: try plain number (no SD given)
    try:
        return float(str(cell).replace(",", ".")), np.nan
    except ValueError:
        return np.nan, np.nan

# ── 4. Process each sheet ────────────────────────────────────────────────────
NUMERIC_COLS = [
    "age", "disease_duration_months", "SNAP_sural",
    "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm",
]

PERIODICITY_RAW = {
    "Caspr1": "Caspr1 periodicity , mean (±SD) ",
    "PanNF":  "PanNF",
}

frames = {}
for sheet_name, df in sheets_raw.items():
    if sheet_name not in PERIODICITY_RAW:
        continue

    # drop unnamed helper columns
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # rename known columns
    df = df.rename(columns=COL_MAP)

    # fix group column: strip text annotations like "1 (SFN)"
    df["group"] = df["group"].astype(str).str.extract(r"(\d)")[0].astype(float)

    # fix sex: binary encode
    df["sex_male"] = (df["sex"].astype(str).str.lower().str.strip() == "male").astype(int)

    # fix numeric columns with possible comma-decimals
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = df[col].apply(fix_decimal)

    # parse periodicity mean ± SD
    raw_col = PERIODICITY_RAW[sheet_name]
    if raw_col in df.columns:
        parsed = df[raw_col].apply(parse_mean_sd)
        df["acf_peak_mean_nm"]  = [p[0] for p in parsed]
        df["acf_peak_sd_nm"]    = [p[1] for p in parsed]
        df["acf_peak_cv"]       = df["acf_peak_sd_nm"] / df["acf_peak_mean_nm"]  # coefficient of variation
        df = df.drop(columns=[raw_col])

    frames[sheet_name] = df
    print(f"\n=== {sheet_name}: {df.shape[0]} rows, {df.shape[1]} cols ===")
    print(df[["patient_id", "group", "age", "acf_peak_mean_nm", "acf_peak_sd_nm"]].to_string(index=False))

# ── 5. Combine for joint analysis ────────────────────────────────────────────
for name, df in frames.items():
    df["marker"] = name

combined = pd.concat(frames.values(), ignore_index=True)

# ── 6. Spearman correlation: ACF peak mean vs. clinical variables ────────────
CLINICAL = ["age", "disease_duration_months", "SNAP_sural",
            "HbA1c_pct", "NIS_LL", "SAS", "NPSI", "IENFD_per_mm"]

print("\n\n=== Spearman correlations: acf_peak_mean_nm vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)

results = []
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_mean_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_mean_nm"], sub[clin])
        results.append(dict(
            marker=sheet_name, variable=clin,
            rho=rho, pval=pval, n=len(sub)
        ))
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

results_df = pd.DataFrame(results)

# ── 7. Why SD/variance of the ACF peak matters ───────────────────────────────
print("""
=============================================================
WHY VARIANCE (SD) OF THE ACF PEAK DISTANCE MATTERS
=============================================================

1. BIOLOGICAL NOISE vs. TRUE SIGNAL
   The SD you already stored reflects within-axon variability of
   node-of-Ranvier spacing.  A large SD means the periodicity is
   irregular — pathological remodelling, demyelination, or
   measurement noise.  Using only the mean ignores this.

2. CORRELATION POWER
   If two patients share the same mean periodicity but very different
   SDs, their nerve physiology differs.  Regressing mean alone against,
   e.g., NIS-LL will leave variance-driven variance unexplained,
   inflating residuals and reducing statistical power.

3. SEPARATE PREDICTORS
   acf_peak_sd_nm (or CV = SD/mean) is itself a clinical correlate.
   It may correlate with disease severity (NIS-LL, NPSI) independently
   of the mean, because irregular spacing disrupts saltatory conduction
   even when average spacing is normal.

4. WEIGHTED CORRELATIONS
   If SD differs greatly across patients, Pearson/Spearman on the mean
   is biased toward high-SD patients.  Weighting each observation by
   1/SD² (inverse-variance weighting) gives a more principled estimate.

5. MIXED-EFFECTS / RELIABILITY MODELS
   In proper modelling the SD feeds into the within-subject variance
   component.  Without it you cannot distinguish:
       • true inter-patient differences in periodicity
       • from measurement unreliability

RECOMMENDATION:
   → Always keep (mean, SD) as two separate numeric columns (done above).
   → Run correlations for acf_peak_sd_nm and acf_peak_cv in addition
     to acf_peak_mean_nm (see Section 8 below).
   → Consider a weighted Spearman or a linear model with 1/SD² weights.
=============================================================
""")

# ── 8. Correlation: ACF peak SD and CV vs. clinical variables ────────────────
print("=== Spearman: acf_peak_sd_nm  vs. clinical variables ===")
print(f"{'Variable':<28} {'Sheet':<8} {'rho':>7} {'p-value':>9} {'n':>4}")
print("-" * 62)
for sheet_name, df in frames.items():
    for clin in CLINICAL:
        sub = df[["acf_peak_sd_nm", clin]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = stats.spearmanr(sub["acf_peak_sd_nm"], sub[clin])
        sig = " *" if pval < 0.05 else ("  †" if pval < 0.10 else "")
        print(f"{clin:<28} {sheet_name:<8} {rho:>7.3f} {pval:>9.4f} {len(sub):>4}{sig}")

# ── 9. Heatmap of Spearman rho — Group 1+2 combined, reliable pairs highlighted
# ─────────────────────────────────────────────────────────────────────────────
# Changes vs previous version:
#   - acf_peak_cv REMOVED (not interpretable without context)
#   - Labels made intuitive: acf_peak_mean_nm → "Periodicity mean (nm)"
#                            acf_peak_sd_nm   → "Periodicity SD (nm)"
#   - Label alignment fixed (ha="right", rotation=45 consistent)
#   - IENFD correlation checked for significance
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# Human-readable axis labels for heatmap
LABEL_MAP = {
    "age":                      "Age (years)",
    "disease_duration_months":  "Disease duration (months)",
    "SNAP_sural":               "SNAP sural nerve (µV)",
    "HbA1c_pct":                "HbA1c (%)",
    "NIS_LL":                   "NIS-LL score",
    "SAS":                      "SAS score",
    "NPSI":                     "NPSI score",
    "IENFD_per_mm":             "IENFD (fibres/mm)",
    "acf_peak_mean_nm":         "Periodicity mean (nm)",
    "acf_peak_sd_nm":           "Periodicity SD (nm)",
}

RELIABLE = {
    "Caspr1": [
        ("acf_peak_mean_nm", "NPSI"),  ("NPSI", "acf_peak_mean_nm"),
        ("acf_peak_mean_nm", "SAS"),   ("SAS",  "acf_peak_mean_nm"),
    ],
    "PanNF": [
        ("acf_peak_sd_nm", "NIS_LL"), ("NIS_LL", "acf_peak_sd_nm"),
    ],
}

for sheet_name, df in frames.items():
    # acf_peak_cv REMOVED
    cols_for_heatmap = CLINICAL + ["acf_peak_mean_nm", "acf_peak_sd_nm"]
    sub = df[df["group"].isin([1.0, 2.0])][[
        c for c in cols_for_heatmap if c in df.columns
    ]].dropna(how="all")
    if sub.shape[0] < 5:
        continue

    corr_matrix = sub.corr(method="spearman", numeric_only=True)
    cols = corr_matrix.columns.tolist()

    # p-value matrix
    pval_matrix = pd.DataFrame(np.ones((len(cols), len(cols))), index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            if i != j:
                pair = sub[[c1, c2]].dropna()
                if len(pair) >= 5:
                    _, p = stats.spearmanr(pair[c1], pair[c2])
                    pval_matrix.loc[c1, c2] = p

    # annotation: rho + star
    annot_matrix = pd.DataFrame("", index=cols, columns=cols)
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            r = corr_matrix.loc[c1, c2]
            p = pval_matrix.loc[c1, c2]
            if i == j:
                annot_matrix.loc[c1, c2] = "1.00"
            else:
                star = "*" if p < 0.05 else ("†" if p < 0.10 else "")
                annot_matrix.loc[c1, c2] = f"{r:.2f}{star}"

    # apply human-readable labels
    readable_cols = [LABEL_MAP.get(c, c) for c in cols]
    corr_matrix_plot = corr_matrix.copy()
    corr_matrix_plot.index   = readable_cols
    corr_matrix_plot.columns = readable_cols
    annot_plot = annot_matrix.copy()
    annot_plot.index   = readable_cols
    annot_plot.columns = readable_cols

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        corr_matrix_plot,
        annot=annot_plot, fmt="", center=0,
        cmap="RdBu_r", vmin=-1, vmax=1,
        linewidths=0.5, ax=ax,
        annot_kws={"size": 8.5},
        cbar_kws={"shrink": 0.8, "pad": 0.02},
    )

    # green borders on reliable cells (using readable labels)
    readable_reliable = [
        (LABEL_MAP.get(rv, rv), LABEL_MAP.get(cv, cv))
        for rv, cv in RELIABLE.get(sheet_name, [])
    ]
    for (row_lbl, col_lbl) in readable_reliable:
        if row_lbl in readable_cols and col_lbl in readable_cols:
            ax.add_patch(plt.Rectangle(
                (readable_cols.index(col_lbl), readable_cols.index(row_lbl)), 1, 1,
                fill=False, edgecolor="#00aa00", lw=3.5, zorder=5
            ))

    ax.set_title(
        f"Spearman ρ matrix — {sheet_name}  (Group 1+2, controls excluded)",
        fontsize=13, fontweight="bold", pad=14
    )
    # fix label alignment
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right",
                       fontsize=9, rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(f"corr_heatmap_{sheet_name}.png", dpi=200, bbox_inches="tight",
                facecolor="white")
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}.png")

    # ── check IENFD significance explicitly ──────────────────────────────────
    print(f"\n  IENFD correlations — {sheet_name} (Group 1+2):")
    for acf_col in ["acf_peak_mean_nm", "acf_peak_sd_nm"]:
        if acf_col in df.columns:
            pair = df[df["group"].isin([1.0, 2.0])][[acf_col, "IENFD_per_mm"]].dropna()
            if len(pair) >= 5:
                rho_i, pval_i = stats.spearmanr(pair[acf_col], pair["IENFD_per_mm"])
                sig_i = "* significant" if pval_i < 0.05 else \
                        ("† trend" if pval_i < 0.10 else "ns — NOT significant")
                print(f"    {LABEL_MAP.get(acf_col, acf_col)} ↔ IENFD: "
                      f"ρ={rho_i:+.3f}  p={pval_i:.4f}  {sig_i}")

    # ── separate legend PNG ───────────────────────────────────────────────────
    fig_leg, ax_leg = plt.subplots(figsize=(6.5, 0.9))
    ax_leg.axis("off")
    legend_items = [
        mpatches.Patch(facecolor="none", edgecolor="#00aa00", linewidth=2.5,
                       label="Reliable correlation (p<0.05, Group 1+2)"),
        mpatches.Patch(facecolor="white", edgecolor="none",
                       label="* p<0.05   † p<0.10"),
    ]
    ax_leg.legend(handles=legend_items, loc="center", ncol=2,
                  fontsize=11, frameon=True, framealpha=1,
                  edgecolor="#cccccc", handlelength=1.5)
    fig_leg.savefig(f"corr_heatmap_{sheet_name}_legend.png", dpi=200,
                    bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: corr_heatmap_{sheet_name}_legend.png")


# ── 10. Publication 3-panel scatter — Group 1 only (diabetic neuropathy) ──────
# Per reviewer feedback:
#   - Group 1 only — Group 2 (blue squares) behaves differently and is excluded
#   - Re-run Spearman within Group 1 only
#   - Labels simplified and made intuitive
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 14, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "axes.linewidth": 1.2, "xtick.major.width": 1.2, "ytick.major.width": 1.2,
    "xtick.major.size": 5,  "ytick.major.size": 5,
})

GROUP_COLORS  = {1.0: "#c0392b", 2.0: "#2980b9"}
GROUP_MARKERS = {1.0: "o",       2.0: "s"}
GROUP_LABELS  = {1.0: "Diabetic neuropathy",
                 2.0: "Diabetes w/o neuropathy"}

PANELS = [
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="NPSI",
         x_label="Caspr1 periodicity mean (nm)",
         y_label="Neuropathic pain\n(NPSI score)", label="A"),
    dict(sheet="Caspr1", x="acf_peak_mean_nm", y="SAS",
         x_label="Caspr1 periodicity mean (nm)",
         y_label="Anxiety / sensory\n(SAS score)", label="B"),
    dict(sheet="PanNF",  x="acf_peak_sd_nm",  y="NIS_LL",
         x_label="PanNF periodicity variability (nm)",
         y_label="Neurological deficit\n(NIS-LL score)", label="C"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.subplots_adjust(wspace=0.42)

for ax, p in zip(axes, PANELS):
    df_sheet = frames[p["sheet"]]

    # ── Group 1 only for regression + stats ──────────────────────────────────
    sub_g1 = df_sheet[df_sheet["group"] == 1.0][[p["x"], p["y"]]].dropna()
    x1 = sub_g1[p["x"]].values.astype(float)
    y1 = sub_g1[p["y"]].values.astype(float)

    rho, pval = stats.spearmanr(x1, y1)
    p_str = f"p = {pval:.4f}" if pval >= 0.001 else f"p = {pval:.2e}"
    sig   = " *" if pval < 0.05 else (" †" if pval < 0.10 else "")
    annot = f"Spearman ρ = {rho:+.3f}{sig}\n{p_str}  (n = {len(x1)})"

    # OLS regression + bootstrap CI — Group 1 only
    slope, intercept, *_ = stats.linregress(x1, y1)
    x_line = np.linspace(x1.min(), x1.max(), 300)
    y_line = slope * x_line + intercept
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(3000):
        idx_b = rng.integers(0, len(x1), len(x1))
        if len(np.unique(x1[idx_b])) < 2: continue
        s, i, *_ = stats.linregress(x1[idx_b], y1[idx_b])
        boots.append(s * x_line + i)
    boot_y = np.array(boots)
    ax.fill_between(x_line, np.percentile(boot_y, 2.5, axis=0),
                    np.percentile(boot_y, 97.5, axis=0),
                    alpha=0.18, color="#888888", zorder=1)
    ax.plot(x_line, y_line, color="#333333", lw=2, ls="--", zorder=2)

    # ── plot Group 1 only ─────────────────────────────────────────────────────
    sub_all = df_sheet[df_sheet["group"] == 1.0][[p["x"], p["y"], "group"]].dropna()
    for grp_val, grp_df in sub_all.groupby("group"):
        ax.scatter(grp_df[p["x"]], grp_df[p["y"]],
                   color=GROUP_COLORS.get(grp_val, "#aaa"),
                   marker=GROUP_MARKERS.get(grp_val, "o"),
                   s=80, edgecolors="white", linewidths=0.7, zorder=3,
                   label=GROUP_LABELS.get(grp_val))

    # stat annotation box
    box_fc = "#fff0f0" if pval < 0.05 else "white"
    ax.text(0.97, 0.04, annot, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc=box_fc, ec="#aaaaaa", lw=1.2))

    # panel letter
    ax.text(-0.15, 1.04, p["label"], transform=ax.transAxes,
            fontsize=18, fontweight="bold", va="top")

    ax.set_xlabel(p["x_label"], labelpad=8)
    ax.set_ylabel(p["y_label"], labelpad=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    xpad = (x1.max() - x1.min()) * 0.08
    ypad = (y1.max() - y1.min()) * 0.10
    ax.set_xlim(x1.min() - xpad, x1.max() + xpad)
    ax.set_ylim(y1.min() - ypad, y1.max() + ypad * 2.5)

fig.suptitle(
    "ACF Periodicity Correlates with Clinical Severity\n"
    "(Diabetic neuropathy — Group 1 only, controls excluded)",
    fontsize=14, fontweight="bold", y=1.04
)
plt.savefig("acf_publication_figure.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig("acf_publication_figure.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: acf_publication_figure.png / .pdf")

# ── separate scatter legend ───────────────────────────────────────────────────
scatter_legend_handles = [
    mpatches.Patch(color=GROUP_COLORS[1.0], label=GROUP_LABELS[1.0]),
    Line2D([0],[0], color="#333333", lw=2, ls="--", label="OLS regression (Group 1)"),
    mpatches.Patch(color="#888888", alpha=0.3, label="95 % CI (bootstrap, n=3,000)"),
]
fig_leg2, ax_leg2 = plt.subplots(figsize=(8, 0.75))
ax_leg2.axis("off")
ax_leg2.legend(handles=scatter_legend_handles, loc="center", ncol=3,
               fontsize=11, frameon=True, framealpha=1, edgecolor="#cccccc",
               handlelength=1.8, handletextpad=0.7, columnspacing=1.8)
fig_leg2.savefig("acf_publication_figure_legend.png", dpi=300,
                 bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: acf_publication_figure_legend.png  (add in Photoshop)")

# ── 11. Export harmonised table ───────────────────────────────────────────────
with pd.ExcelWriter("acf_harmonised.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in frames.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    combined.to_excel(writer, sheet_name="combined", index=False)

print("\nHarmonised file written: acf_harmonised.xlsx")
print("* (p<0.05)   † (p<0.10)")

In [ ]:
import os
print("Current working directory:", os.getcwd())
print("\nFiles saved here:")
for f in os.listdir(os.getcwd()):
    if any(f.endswith(ext) for ext in ['.png', '.pdf', '.xlsx', '.csv']):
        print(f"  {f}")